In [4]:
# 代码作用：读取LIMS数据汇总为 榨季期分蜜指标统计报表v1
# 核心逻辑：全局累计合格率（按月度、班次独立累计）
# 2026/04/02
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType, DateType,DoubleType
# 关键：明确导入Spark函数，避免和Python内置函数混淆
from pyspark.sql.functions import (
    current_timestamp, split, lit, col, row_number, sum as spark_sum, 
    round as spark_round, avg, when, to_date, count, coalesce
)
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()



# ===================== 1. 读取LIMS班报 成品糖产量 数据 =====================
lims_cpt_sql = """
SELECT 
    season,
    niandudm,
    gongsidm,
    rq,
    banbiedm,
    name,
    SUM(chanliang) AS chanliang
FROM (
    SELECT 
        CONCAT(
            CAST((CAST(niandudm AS INTEGER) - 1) % 100 AS STRING), 
            '/', 
            CAST(CAST(niandudm AS INTEGER) % 100 AS STRING)
        ) AS season,
        niandudm,
        gongsidm,
        rq,
        CASE 
            WHEN banbiedm = '01' THEN '甲班'
            WHEN banbiedm = '02' THEN '乙班'
            WHEN banbiedm = '03' THEN '丙班'
            ELSE COALESCE(banbiedm, '未知班别')
        END AS banbiedm,
        name,
        chanliang
    FROM dwr.dwr_lims_sugar_product_detai_f_1d 
    WHERE gongsidm = 'FNR' 
        -- AND name = '原糖'
        -- AND name = '金砂糖'
) t
GROUP BY season, niandudm, gongsidm, rq, banbiedm, name
ORDER BY rq DESC;
"""
lims_cpt_df = spark.sql(lims_cpt_sql)

# ===================== 2. 读取LIMS 精糖班报实绩 数据 =====================
lims_bbsj_sql = """
SELECT 
    CONCAT(SUBSTR(CAST(year - 1 AS STRING), 3, 2), '/', SUBSTR(CAST(year AS STRING), 3, 2)) AS season,
    company,
    date,
    CASE 
        WHEN banbiedm = '01' THEN '甲班'
        WHEN banbiedm = '02' THEN '乙班'
        WHEN banbiedm = '03' THEN '丙班'
        WHEN banbiedm = '04' THEN '丁班'
        ELSE COALESCE(banbiedm, '未知班别')
    END AS banbiedm,
    hrtcl
FROM dwr.dwr_lims_sugar_refined_shift_actual_f_1d 
WHERE company = 'FNR'
"""
lims_bbsj_df = spark.sql(lims_bbsj_sql)


# ===================== 3. 读取LIMS 榨蔗班报实绩 数据 =====================
lims_zzsj_sql = """
SELECT 
season,
tb_gongsidm,
tb_rq,
CASE 
    WHEN tb_banbiedm = '01' THEN '甲班'
    WHEN tb_banbiedm = '02' THEN '乙班'
    WHEN tb_banbiedm = '03' THEN '丙班'
    ELSE COALESCE(tb_banbiedm, '未知班别')
END AS banbie_name,
sj_zhazheliang
FROM dwd.dwd_lims_batch_report_actual_data_f_1h WHERE tb_gongsidm = 'FNR';
"""
lims_zzsj_df = spark.sql(lims_zzsj_sql)



# ===================== 4. 读取LIMS样品分析 金砂糖 数据 =====================
lims_jst_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '金砂糖'
        AND test_name1 IN ('产量', '色值', '干燥失重', '糖度')
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '产量' THEN test_value END) AS golden_sugar_yield,        -- 金砂糖产量
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS golden_sugar_color_value, -- 金砂糖色值
    MAX(CASE WHEN test_name1 = '干燥失重' THEN test_value END) AS golden_sugar_dry_loss, -- 金砂糖干燥失重
    MAX(CASE WHEN test_name1 = '糖度' THEN test_value END) AS golden_sugar_brix          -- 金砂糖糖度
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '产量' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '干燥失重' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '糖度' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_jst_df = spark.sql(lims_jst_sql)


# ===================== 5. 读取LIMS样品分析 原糖 数据 =====================
lims_yt_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '原糖'
        AND test_name1 IN ('色值', '干燥失重', '产量')   -- 增加产量
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS raw_sugar_color_value,   -- 原糖色值
    MAX(CASE WHEN test_name1 = '干燥失重' THEN test_value END) AS raw_sugar_dry_loss,  -- 原糖干燥失重
    MAX(CASE WHEN test_name1 = '产量' THEN test_value END) AS raw_sugar_yield          -- 原糖产量
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '干燥失重' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '产量' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_yt_df = spark.sql(lims_yt_sql)


# ===================== 6. 读取LIMS 洄溶糖浆 数据 =====================
lims_hrtj_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '炼糖生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '洄溶糖浆'
        AND test_name1 IN ('锤度', '色值')
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS remelt_syrup_brix,    -- 洄溶糖浆锤度
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS remelt_syrup_color_value  -- 洄溶糖浆色值
FROM raw_data 
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_hrtj_df = spark.sql(lims_hrtj_sql)


# ===================== 7. 读取LIMS样品分析 丙膏 数据 =====================
lims_bg_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '丙膏'                    -- 物料改为丙膏
        AND test_name1 IN ('视纯度')                   -- 仅检测视纯度
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS propyl_paste_apparent_purity  -- 丙膏视纯度
FROM raw_data 
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_bg_df = spark.sql(lims_bg_sql)


# ===================== 8. 读取LIMS样品分析 丙糖糊 数据 =====================
lims_bth_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '丙糖糊'
        AND test_name1 IN ('色值', '锤度', '视纯度')          -- 增加视纯度
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS third_sugar_paste_color_value,   -- 丙糖糊色值
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS third_sugar_paste_brix,         -- 丙糖糊锤度
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS third_sugar_paste_apparent_purity  -- 丙糖糊视纯度
FROM raw_data 
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_bth_df = spark.sql(lims_bth_sql)

# ===================== 9. 读取LIMS样品分析 榨蔗桔水 数据 =====================
lims_zzjs_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '榨蔗桔水'                              -- 物料改为榨蔗桔水
        AND test_name1 IN ('重力纯度', '产量', '锤度', '视纯度')      -- 四个检测项目
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '重力纯度' THEN test_value END) AS cane_molasses_gravity_purity,   -- 榨蔗桔水重力纯度
    MAX(CASE WHEN test_name1 = '产量' THEN test_value END) AS cane_molasses_yield,                -- 榨蔗桔水产量
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS cane_molasses_brix,                -- 榨蔗桔水锤度
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS cane_molasses_apparent_purity     -- 榨蔗桔水视纯度
FROM raw_data 
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '重力纯度' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '产量' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_zzjs_df = spark.sql(lims_zzjs_sql)
# ===================== 10. 读取LIMS样品分析 合格率标准 数据 =====================
lims_hgbz_sql = """
SELECT * FROM dwd.dwd_lims_sugar_quality_standard_f_1h WHERE org_code = 'FNR'
"""
lims_hgbz_df = spark.sql(lims_hgbz_sql)

# ===================== 11. 读取LIMS样品分析 金砂糖膏 数据 =====================
lims_jstg_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        -- 保留原始文本值（用于备注）
        test_test_value AS test_value_raw,
        -- 仅对视纯度和锤度转换为数值
        CASE 
            WHEN test_name1 IN ('视纯度', '锤度') AND test_test_value REGEXP '^[0-9.]+$' 
                THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value_num
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '甲膏'                     -- 样品改为甲膏
        AND test_name1 IN ('备注', '视纯度', '锤度')       -- 检测项目改为备注、视纯度、锤度
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '备注' THEN test_value_raw END) AS remark,   -- 备注（文本）
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value_num END) AS purity,   -- 视纯度（数值）
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value_num END) AS brix      -- 锤度（数值）
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '备注' THEN test_value_raw END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '视纯度' THEN test_value_num END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '锤度' THEN test_value_num END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_jstg_df = spark.sql(lims_jstg_sql)
lims_jstg_df.show()

Spark 启动中...


/opt/tiger/spark/python/pyspark/context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning


+-------+---------------+-----------+----------+---------------+--------------+------+------+-----+
|company|crushing_season|record_date|work_shift|ord_sample_name|        ord_id|remark|purity| brix|
+-------+---------------+-----------+----------+---------------+--------------+------+------+-----+
|    FNR|          25/26| 2026-04-13|      甲班|           甲膏|81799532788608|      | 83.26|93.66|
|    FNR|          25/26| 2026-04-13|      甲班|           甲膏|81824211188608|  null| 85.08|93.54|
|    FNR|          25/26| 2026-04-13|      甲班|           甲膏|81814164110208|  null| 85.19|93.06|
|    FNR|          25/26| 2026-04-13|      乙班|           甲膏|81856733019008|  null| 84.67|93.84|
|    FNR|          25/26| 2026-04-13|      甲班|           甲膏|81824164494208|  null| 84.18|92.94|
|    FNR|          25/26| 2026-04-12|      丙班|           甲膏|81784400526208|  null| 82.66|93.84|
|    FNR|          25/26| 2026-04-12|      丙班|           甲膏|81775372942208|  null| 82.18|95.76|
|    FNR|          25/26| 20

In [5]:
# ================================================================
# ===================== 柱状图数据：按榨季+班次聚合（数值版） =====================
# ================================================================

from pyspark.sql.functions import month, concat, lit, sum as spark_sum, round as spark_round, avg, count, when, col
from pyspark.sql import DataFrame
from pyspark.sql.types import DoubleType
from functools import reduce

# ---------- 辅助函数：按榨季+班次聚合数值型指标（如产量、纯度） ----------
def aggregate_season_shift_value(df, group_cols, value_col, indicator_name, 
                                 agg_func='sum', has_total=True):
    """
    返回 indicator_value 为 Double 类型
    """
    agg_expr = spark_sum(value_col) if agg_func == 'sum' else avg(value_col)
    by_shift = df.groupBy(*group_cols).agg(agg_expr.alias("val")) \
        .withColumn("indicator_value", spark_round(col("val"), 2).cast(DoubleType())) \
        .withColumn("month_period", lit("榨季累计")) \
        .withColumn("indicator_name", lit(indicator_name)) \
        .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")
    
    if not has_total:
        return by_shift
    
    total = df.groupBy("season", "company").agg(agg_expr.alias("val")) \
        .withColumn("indicator_value", spark_round(col("val"), 2).cast(DoubleType())) \
        .withColumn("shift", lit("累计")) \
        .withColumn("month_period", lit("榨季累计")) \
        .withColumn("indicator_name", lit(indicator_name)) \
        .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")
    
    return by_shift.unionByName(total)

# ---------- 辅助函数：按榨季+班次计算合格率指标（返回小数，如0.95） ----------
def aggregate_season_shift_pass_rate(df, group_cols, condition_col, indicator_name, has_total=True):
    """
    合格率以小数形式返回，保留4位小数（如0.9500）
    """
    by_shift = df.groupBy(*group_cols) \
        .agg(spark_sum("is_qualified").alias("qualified"), count("*").alias("total")) \
        .withColumn("rate", when(col("total")>0, col("qualified")/col("total")).otherwise(0.0)) \
        .withColumn("indicator_value", spark_round(col("rate"), 4).cast(DoubleType())) \
        .withColumn("month_period", lit("榨季累计")) \
        .withColumn("indicator_name", lit(indicator_name)) \
        .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")
    
    if not has_total:
        return by_shift
    
    total = df.groupBy("season", "company") \
        .agg(spark_sum("is_qualified").alias("qualified"), count("*").alias("total")) \
        .withColumn("rate", when(col("total")>0, col("qualified")/col("total")).otherwise(0.0)) \
        .withColumn("indicator_value", spark_round(col("rate"), 4).cast(DoubleType())) \
        .withColumn("shift", lit("累计")) \
        .withColumn("month_period", lit("榨季累计")) \
        .withColumn("indicator_name", lit(indicator_name)) \
        .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")
    
    return by_shift.unionByName(total)

# ---------- 纯度差计算（返回数值） ----------
def compute_purity_diff_season_shift(propyl_purity_df, cane_purity_df):
    # propyl_purity_df 和 cane_purity_df 的 indicator_value 已经是 Double 类型
    propyl = propyl_purity_df.withColumnRenamed("indicator_value", "propyl_val") \
        .select("season", "company", "shift", "propyl_val")
    cane = cane_purity_df.withColumnRenamed("indicator_value", "cane_val") \
        .select("season", "company", "shift", "cane_val")
    
    by_shift = propyl.join(cane, ["season", "company", "shift"]) \
        .withColumn("diff_val", col("propyl_val") - col("cane_val")) \
        .withColumn("indicator_value", spark_round(col("diff_val"), 2).cast(DoubleType())) \
        .withColumn("month_period", lit("榨季累计")) \
        .withColumn("indicator_name", lit("丙膏桔水纯度差（AP）")) \
        .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")
    
    # 累计
    propyl_total = propyl_purity_df.filter(col("shift")=="累计") \
        .withColumnRenamed("indicator_value", "propyl_val") \
        .select("season", "company", "propyl_val")
    cane_total = cane_purity_df.filter(col("shift")=="累计") \
        .withColumnRenamed("indicator_value", "cane_val") \
        .select("season", "company", "cane_val")
    
    total = propyl_total.join(cane_total, ["season", "company"]) \
        .withColumn("diff_val", col("propyl_val") - col("cane_val")) \
        .withColumn("indicator_value", spark_round(col("diff_val"), 2).cast(DoubleType())) \
        .withColumn("shift", lit("累计")) \
        .withColumn("month_period", lit("榨季累计")) \
        .withColumn("indicator_name", lit("丙膏桔水纯度差（AP）")) \
        .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")
    
    return by_shift.unionByName(total)


# ===================== 1. 读取目标值表并转换为长格式 =====================
target_df = spark.sql("""
    SELECT 
        '原糖产量（t）' AS indicator_name,
        CAST(raw_sugar AS DOUBLE) AS indicator_value
    FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '金砂糖产量（t）', CAST(golden_sugar AS DOUBLE) FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '产糖率（%）', (raw_sugar + golden_sugar) / sugar_cane_crushing FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '入库原糖量（t）', CAST(stored_raw_sugar AS DOUBLE) FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '回溶糖量（t）', CAST(refined_dissolved_raw_sugar AS DOUBLE) FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '原糖色值合格率（%）', 0.95 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '原糖水份合格率（%）', 0.95 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '回溶糖浆锤度合格率（%）', 0.95 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '回溶糖浆色值合格率（%）', 0.90 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '金砂糖糖膏纯度合格率（%）', 1.00 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '金砂糖糖膏锤度合格率（%）', 1.00 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '金砂糖色值合格率（%）', 1.00 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '金砂糖水份合格率（%）', 1.00 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '金砂糖糖度合格率（%）', 1.00 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '桔水产率（%）', molasses_yield_rate FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '丙膏平均纯度（AP）', 0.00 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '桔水平均纯度（AP）', 0.00 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '丙膏桔水纯度差（AP）', 17.00 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '丙糖糊色值合格率（%）', 0.90 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '丙糖糊锤度合格率（%）', 0.90 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '桔水重力纯度合格率（%）', 0.90 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
    UNION ALL
    SELECT '桔水锤度合格率（%）', 0.90 FROM app.app_lims_zhajimubiao_f_1d WHERE factory_code = 'FNR'
""")

# 关联榨季和公司
season_company_df = lims_cpt_df.select("season", "gongsidm").distinct() \
    .withColumnRenamed("gongsidm", "company")
if season_company_df.count() == 0:
    season_company_df = spark.createDataFrame([("24/25", "FNR")], ["season", "company"])

target_df = target_df.crossJoin(season_company_df) \
    .withColumn("shift", lit("目标")) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_value", spark_round(col("indicator_value"), 2).cast(DoubleType())) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")


# ===================== 2. 准备基础数据 =====================
sugar_product_df = lims_cpt_df \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "name", "chanliang")

crushing_season_shift_df = lims_zzsj_df \
    .withColumnRenamed("tb_gongsidm", "company") \
    .withColumnRenamed("banbie_name", "shift") \
    .groupBy("season", "company", "shift") \
    .agg(spark_sum("sj_zhazheliang").alias("total_crushing"))

remelt_df = lims_bbsj_df \
    .withColumnRenamed("banbiedm", "shift") \
    .groupBy("season", "company", "shift") \
    .agg(spark_sum("hrtcl").alias("total_remelt"))

golden_df = lims_jst_df \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("work_shift", "shift") \
    .select("season", "company", "shift", "golden_sugar_yield", "golden_sugar_color_value",
            "golden_sugar_dry_loss", "golden_sugar_brix")

raw_df = lims_yt_df \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("work_shift", "shift") \
    .select("season", "company", "shift", "raw_sugar_yield", "raw_sugar_color_value", "raw_sugar_dry_loss")

remelt_syrup_df = lims_hrtj_df \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("work_shift", "shift") \
    .select("season", "company", "shift", "remelt_syrup_brix", "remelt_syrup_color_value")

propyl_paste_df = lims_bg_df \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("work_shift", "shift") \
    .select("season", "company", "shift", "propyl_paste_apparent_purity")

third_paste_df = lims_bth_df \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("work_shift", "shift") \
    .select("season", "company", "shift", "third_sugar_paste_color_value",
            "third_sugar_paste_brix", "third_sugar_paste_apparent_purity")

cane_molasses_df = lims_zzjs_df \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("work_shift", "shift") \
    .select("season", "company", "shift", "cane_molasses_gravity_purity", "cane_molasses_yield",
            "cane_molasses_brix", "cane_molasses_apparent_purity")

golden_paste_df = lims_jstg_df \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("work_shift", "shift") \
    .select("season", "company", "shift", "remark", "purity", "brix")


# ===================== 3. 计算各指标的实际值 =====================
def get_standard(material, item):
    row = lims_hgbz_df.filter((col("material_name")==material) & (col("item_name")==item)) \
        .orderBy(col("dismonth_date").desc()).limit(1).collect()
    if row:
        return row[0]["lower_limit"], row[0]["upper_limit"]
    return None, None

# 指标2：金砂糖产量
golden_yield_shift = aggregate_season_shift_value(
    sugar_product_df.filter(col("name")=="金砂糖"),
    ["season", "company", "shift"], "chanliang", "金砂糖产量（t）", has_total=True
)

# 指标3：产糖率
total_sugar_shift = sugar_product_df.groupBy("season", "company", "shift") \
    .agg(spark_sum("chanliang").alias("total_sugar"))
sugar_rate_shift = total_sugar_shift.join(crushing_season_shift_df, ["season", "company", "shift"], "inner") \
    .withColumn("rate", col("total_sugar") / col("total_crushing")) \
    .withColumn("indicator_value", spark_round(col("rate"), 4).cast(DoubleType())) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("产糖率（%）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")

total_sugar_total = sugar_product_df.groupBy("season", "company") \
    .agg(spark_sum("chanliang").alias("total_sugar"))
crushing_total = crushing_season_shift_df.groupBy("season", "company") \
    .agg(spark_sum("total_crushing").alias("total_crushing"))
sugar_rate_total = total_sugar_total.join(crushing_total, ["season", "company"]) \
    .withColumn("rate", col("total_sugar") / col("total_crushing")) \
    .withColumn("indicator_value", spark_round(col("rate"), 4).cast(DoubleType())) \
    .withColumn("shift", lit("累计")) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("产糖率（%）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")
sugar_rate_shift = sugar_rate_shift.unionByName(sugar_rate_total)

# 指标4：入库原糖量（直接汇总原糖产量）
raw_storage_shift = aggregate_season_shift_value(
    sugar_product_df.filter(col("name")=="原糖"),
    ["season", "company", "shift"], "chanliang", "入库原糖量（t）", has_total=True
)

# 指标5：回溶糖量
remelt_shift = aggregate_season_shift_value(
    remelt_df, ["season", "company", "shift"], "total_remelt", "回溶糖量（t）", has_total=True
)

# 指标1：原糖产量 = 入库原糖量 + 回溶糖量
storage_for_sum = raw_storage_shift.select(
    "season", "company", "shift", "month_period",
    col("indicator_value").alias("storage_val")
)
remelt_for_sum = remelt_shift.select(
    "season", "company", "shift", "month_period",
    col("indicator_value").alias("remelt_val")
)
raw_yield_shift = storage_for_sum.join(remelt_for_sum,
                                       on=["season", "company", "shift", "month_period"],
                                       how="full") \
    .fillna(0) \
    .withColumn("indicator_value", spark_round(col("storage_val") + col("remelt_val"), 2).cast(DoubleType())) \
    .select("season", "company", "shift", "month_period", "indicator_value") \
    .withColumn("indicator_name", lit("原糖产量（t）"))

# 指标6：原糖色值合格率
low_c, up_c = get_standard("原糖", "色值")
raw_color_qualified = raw_df.filter(col("raw_sugar_color_value").isNotNull()) \
    .withColumn("is_qualified", when((col("raw_sugar_color_value") >= low_c) & (col("raw_sugar_color_value") <= up_c), 1).otherwise(0))
raw_color_rate_shift = aggregate_season_shift_pass_rate(
    raw_color_qualified, ["season", "company", "shift"], "is_qualified", "原糖色值合格率（%）"
)

# 指标7：原糖水份合格率
low_m, up_m = get_standard("原糖", "干燥失重")
raw_moisture_qualified = raw_df.filter(col("raw_sugar_dry_loss").isNotNull()) \
    .withColumn("is_qualified", when((col("raw_sugar_dry_loss") >= low_m) & (col("raw_sugar_dry_loss") <= up_m), 1).otherwise(0))
raw_moisture_rate_shift = aggregate_season_shift_pass_rate(
    raw_moisture_qualified, ["season", "company", "shift"], "is_qualified", "原糖水份合格率（%）"
)

# 指标8：回溶糖浆锤度合格率
low_rb, up_rb = get_standard("洄溶糖浆", "锤度")
remelt_brix_qualified = remelt_syrup_df.filter(col("remelt_syrup_brix").isNotNull()) \
    .withColumn("is_qualified", when((col("remelt_syrup_brix") >= low_rb) & (col("remelt_syrup_brix") <= up_rb), 1).otherwise(0))
remelt_brix_rate_shift = aggregate_season_shift_pass_rate(
    remelt_brix_qualified, ["season", "company", "shift"], "is_qualified", "回溶糖浆锤度合格率（%）"
)

# 指标9：回溶糖浆色值合格率
low_rc, up_rc = get_standard("洄溶糖浆", "色值")
remelt_color_qualified = remelt_syrup_df.filter(col("remelt_syrup_color_value").isNotNull()) \
    .withColumn("is_qualified", when((col("remelt_syrup_color_value") >= low_rc) & (col("remelt_syrup_color_value") <= up_rc), 1).otherwise(0))
remelt_color_rate_shift = aggregate_season_shift_pass_rate(
    remelt_color_qualified, ["season", "company", "shift"], "is_qualified", "回溶糖浆色值合格率（%）"
)

# 指标10：金砂糖糖膏纯度合格率
golden_paste_purity_qualified = golden_paste_df.filter(col("remark")=="金砂糖") \
    .filter(col("purity").isNotNull()) \
    .withColumn("is_qualified", when((col("purity") >= 70.0) & (col("purity") <= 78.0), 1).otherwise(0))
golden_paste_purity_rate_shift = aggregate_season_shift_pass_rate(
    golden_paste_purity_qualified, ["season", "company", "shift"], "is_qualified", "金砂糖糖膏纯度合格率（%）"
)

# 指标11：金砂糖糖膏锤度合格率
golden_paste_brix_qualified = golden_paste_df.filter(col("remark")=="金砂糖") \
    .filter(col("brix").isNotNull()) \
    .withColumn("is_qualified", when((col("brix") >= 94.0) & (col("brix") <= 99.0), 1).otherwise(0))
golden_paste_brix_rate_shift = aggregate_season_shift_pass_rate(
    golden_paste_brix_qualified, ["season", "company", "shift"], "is_qualified", "金砂糖糖膏锤度合格率（%）"
)

# 指标12：金砂糖色值合格率
low_gc, up_gc = get_standard("金砂糖", "色值")
golden_color_qualified = golden_df.filter(col("golden_sugar_color_value").isNotNull()) \
    .withColumn("is_qualified", when((col("golden_sugar_color_value") >= low_gc) & (col("golden_sugar_color_value") <= up_gc), 1).otherwise(0))
golden_color_rate_shift = aggregate_season_shift_pass_rate(
    golden_color_qualified, ["season", "company", "shift"], "is_qualified", "金砂糖色值合格率（%）"
)

# 指标13：金砂糖水份合格率
low_gm, up_gm = get_standard("金砂糖", "干燥失重")
golden_moisture_qualified = golden_df.filter(col("golden_sugar_dry_loss").isNotNull()) \
    .withColumn("is_qualified", when((col("golden_sugar_dry_loss") >= low_gm) & (col("golden_sugar_dry_loss") <= up_gm), 1).otherwise(0))
golden_moisture_rate_shift = aggregate_season_shift_pass_rate(
    golden_moisture_qualified, ["season", "company", "shift"], "is_qualified", "金砂糖水份合格率（%）"
)

# 指标14：金砂糖糖度合格率
low_gb, up_gb = get_standard("金砂糖", "糖度")
golden_brix_qualified = golden_df.filter(col("golden_sugar_brix").isNotNull()) \
    .withColumn("is_qualified", when((col("golden_sugar_brix") >= low_gb) & (col("golden_sugar_brix") <= up_gb), 1).otherwise(0))
golden_brix_rate_shift = aggregate_season_shift_pass_rate(
    golden_brix_qualified, ["season", "company", "shift"], "is_qualified", "金砂糖糖度合格率（%）"
)

# 指标15：桔水产率
molasses_shift = cane_molasses_df.filter(col("cane_molasses_yield").isNotNull()) \
    .groupBy("season", "company", "shift").agg(spark_sum("cane_molasses_yield").alias("total_molasses"))
molasses_rate_shift = molasses_shift.join(crushing_season_shift_df, ["season", "company", "shift"], "inner") \
    .withColumn("rate", col("total_molasses") / col("total_crushing")*100) \
    .withColumn("indicator_value", spark_round(col("rate"), 4).cast(DoubleType())) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("桔水产率（%）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")

molasses_total = cane_molasses_df.filter(col("cane_molasses_yield").isNotNull()) \
    .groupBy("season", "company").agg(spark_sum("cane_molasses_yield").alias("total_molasses"))
molasses_rate_total = molasses_total.join(crushing_total, ["season", "company"]) \
    .withColumn("rate", col("total_molasses") / col("total_crushing")*100) \
    .withColumn("indicator_value", spark_round(col("rate"), 4).cast(DoubleType())) \
    .withColumn("shift", lit("累计")) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("桔水产率（%）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")
molasses_rate_shift = molasses_rate_shift.unionByName(molasses_rate_total)

# ===================== 指标16：丙膏平均纯度（直接计算，避免中间错误） =====================
propyl_purity_shift = propyl_paste_df.filter(col("propyl_paste_apparent_purity").isNotNull()) \
    .groupBy("season", "company", "shift") \
    .agg(avg("propyl_paste_apparent_purity").alias("avg_val"))
propyl_purity_shift = propyl_purity_shift \
    .withColumn("indicator_value", spark_round(col("avg_val"), 2).cast(DoubleType())) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("丙膏平均纯度（AP）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")

propyl_purity_total = propyl_paste_df.filter(col("propyl_paste_apparent_purity").isNotNull()) \
    .groupBy("season", "company") \
    .agg(avg("propyl_paste_apparent_purity").alias("avg_val")) \
    .withColumn("indicator_value", spark_round(col("avg_val"), 2).cast(DoubleType())) \
    .withColumn("shift", lit("累计")) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("丙膏平均纯度（AP）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")

propyl_purity_shift = propyl_purity_shift.unionByName(propyl_purity_total)

# ===================== 指标17：桔水平均纯度（直接计算） =====================
cane_purity_shift = cane_molasses_df.filter(col("cane_molasses_apparent_purity").isNotNull()) \
    .groupBy("season", "company", "shift") \
    .agg(avg("cane_molasses_apparent_purity").alias("avg_val"))
cane_purity_shift = cane_purity_shift \
    .withColumn("indicator_value", spark_round(col("avg_val"), 2).cast(DoubleType())) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("桔水平均纯度（AP）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")

cane_purity_total = cane_molasses_df.filter(col("cane_molasses_apparent_purity").isNotNull()) \
    .groupBy("season", "company") \
    .agg(avg("cane_molasses_apparent_purity").alias("avg_val")) \
    .withColumn("indicator_value", spark_round(col("avg_val"), 2).cast(DoubleType())) \
    .withColumn("shift", lit("累计")) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("桔水平均纯度（AP）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")

cane_purity_shift = cane_purity_shift.unionByName(cane_purity_total)

# ===================== 指标18：丙膏桔水纯度差（基于上述累计值正确计算） =====================
# 班次纯度差
propyl_shift_avg = propyl_purity_shift.filter(col("shift") != "累计") \
    .withColumnRenamed("indicator_value", "propyl_val") \
    .select("season", "company", "shift", "propyl_val")
cane_shift_avg = cane_purity_shift.filter(col("shift") != "累计") \
    .withColumnRenamed("indicator_value", "cane_val") \
    .select("season", "company", "shift", "cane_val")
purity_diff_shift = propyl_shift_avg.join(cane_shift_avg, ["season", "company", "shift"]) \
    .withColumn("diff_val", col("propyl_val") - col("cane_val")) \
    .withColumn("indicator_value", spark_round(col("diff_val"), 2).cast(DoubleType())) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("丙膏桔水纯度差（AP）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")

# 累计纯度差（直接使用累计纯度相减）
propyl_total_val = propyl_purity_shift.filter(col("shift") == "累计") \
    .withColumnRenamed("indicator_value", "propyl_val") \
    .select("season", "company", "propyl_val")
cane_total_val = cane_purity_shift.filter(col("shift") == "累计") \
    .withColumnRenamed("indicator_value", "cane_val") \
    .select("season", "company", "cane_val")
purity_diff_total = propyl_total_val.join(cane_total_val, ["season", "company"]) \
    .withColumn("diff_val", col("propyl_val") - col("cane_val")) \
    .withColumn("indicator_value", spark_round(col("diff_val"), 2).cast(DoubleType())) \
    .withColumn("shift", lit("累计")) \
    .withColumn("month_period", lit("榨季累计")) \
    .withColumn("indicator_name", lit("丙膏桔水纯度差（AP）")) \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value")

purity_diff_shift = purity_diff_shift.unionByName(purity_diff_total)

# 指标19：丙糖糊色值合格率
b_paste_std = spark.sql("""
    SELECT minimum_value, maximum_value 
    FROM dwd.dwd_mes_process_standard_f_1h 
    WHERE project_name = 'B糖糊色值' 
    ORDER BY create_time DESC LIMIT 1
""").collect()
low_tc, up_tc = b_paste_std[0]["minimum_value"], b_paste_std[0]["maximum_value"]
third_color_qualified = third_paste_df.filter(col("third_sugar_paste_color_value").isNotNull()) \
    .withColumn("is_qualified", when((col("third_sugar_paste_color_value") >= low_tc) & (col("third_sugar_paste_color_value") <= up_tc), 1).otherwise(0))
third_color_rate_shift = aggregate_season_shift_pass_rate(
    third_color_qualified, ["season", "company", "shift"], "is_qualified", "丙糖糊色值合格率（%）"
)

# 指标20：丙糖糊锤度合格率
low_tb, up_tb = get_standard("丙糖糊", "锤度")
third_brix_qualified = third_paste_df.filter(col("third_sugar_paste_brix").isNotNull()) \
    .withColumn("is_qualified", when((col("third_sugar_paste_brix") >= low_tb) & (col("third_sugar_paste_brix") <= up_tb), 1).otherwise(0))
third_brix_rate_shift = aggregate_season_shift_pass_rate(
    third_brix_qualified, ["season", "company", "shift"], "is_qualified", "丙糖糊锤度合格率（%）"
)

# 指标21：桔水重力纯度合格率
low_cg, up_cg = get_standard("榨蔗桔水", "重力纯度")
cane_gravity_qualified = cane_molasses_df.filter(col("cane_molasses_gravity_purity").isNotNull()) \
    .withColumn("is_qualified", when((col("cane_molasses_gravity_purity") >= low_cg) & (col("cane_molasses_gravity_purity") <= up_cg), 1).otherwise(0))
cane_gravity_rate_shift = aggregate_season_shift_pass_rate(
    cane_gravity_qualified, ["season", "company", "shift"], "is_qualified", "桔水重力纯度合格率（%）"
)

# 指标22：桔水锤度合格率
low_cb, up_cb = get_standard("榨蔗桔水", "锤度")
cane_brix_qualified = cane_molasses_df.filter(col("cane_molasses_brix").isNotNull()) \
    .withColumn("is_qualified", when((col("cane_molasses_brix") >= low_cb) & (col("cane_molasses_brix") <= up_cb), 1).otherwise(0))
cane_brix_rate_shift = aggregate_season_shift_pass_rate(
    cane_brix_qualified, ["season", "company", "shift"], "is_qualified", "桔水锤度合格率（%）"
)


# ===================== 4. 合并所有指标（实际值 + 目标值） =====================
all_actual_dfs = [
    raw_yield_shift,
    golden_yield_shift,
    sugar_rate_shift,
    raw_storage_shift,
    remelt_shift,
    raw_color_rate_shift,
    raw_moisture_rate_shift,
    remelt_brix_rate_shift,
    remelt_color_rate_shift,
    golden_paste_purity_rate_shift,
    golden_paste_brix_rate_shift,
    golden_color_rate_shift,
    golden_moisture_rate_shift,
    golden_brix_rate_shift,
    molasses_rate_shift,
    propyl_purity_shift,
    cane_purity_shift,
    purity_diff_shift,
    third_color_rate_shift,
    third_brix_rate_shift,
    cane_gravity_rate_shift,
    cane_brix_rate_shift
]

# 统一类型（确保都是 Double）
def ensure_double(df):
    return df.withColumn("indicator_value", col("indicator_value").cast(DoubleType()))

all_actual_dfs = [ensure_double(df) for df in all_actual_dfs]
final_actual_df = reduce(lambda a, b: a.unionByName(b), all_actual_dfs)

# 合并目标值（已经是 Double）
final_chart_df = final_actual_df.unionByName(target_df)

# 排序预览（可选）
final_chart_df.orderBy("season", "company", "indicator_name", "shift").show(200, truncate=False)

# ===================== 5. 写入Hive表 =====================
target_table = "app.app_lims_season_centrifugal_indicator_chart_f_1h"
spark.sql(f"DROP TABLE {target_table}")
final_chart_df.write.mode("overwrite").saveAsTable(target_table)
print(f"柱状图数据已写入 {target_table}")
spark.stop()
print("Spark已关闭")

+------+-------+-----+------------+---------------+-------------------------+
|season|company|shift|month_period|indicator_value|indicator_name           |
+------+-------+-----+------------+---------------+-------------------------+
|22/23 |FNR    |丙班 |榨季累计    |0.0            |丙糖糊色值合格率（%）    |
|22/23 |FNR    |乙班 |榨季累计    |0.0            |丙糖糊色值合格率（%）    |
|22/23 |FNR    |甲班 |榨季累计    |0.0            |丙糖糊色值合格率（%）    |
|22/23 |FNR    |目标 |榨季累计    |0.9            |丙糖糊色值合格率（%）    |
|22/23 |FNR    |累计 |榨季累计    |0.0            |丙糖糊色值合格率（%）    |
|22/23 |FNR    |丙班 |榨季累计    |0.767          |丙糖糊锤度合格率（%）    |
|22/23 |FNR    |乙班 |榨季累计    |0.858          |丙糖糊锤度合格率（%）    |
|22/23 |FNR    |甲班 |榨季累计    |0.7889         |丙糖糊锤度合格率（%）    |
|22/23 |FNR    |目标 |榨季累计    |0.9            |丙糖糊锤度合格率（%）    |
|22/23 |FNR    |累计 |榨季累计    |0.8045         |丙糖糊锤度合格率（%）    |
|22/23 |FNR    |丙班 |榨季累计    |52.78          |丙膏平均纯度（AP）       |
|22/23 |FNR    |乙班 |榨季累计    |52.36          |丙膏平均纯度（AP）       |
|22/23 |FNR    |甲班

柱状图数据已写入 app.app_lims_season_centrifugal_indicator_chart_f_1h
Spark已关闭
